# 可视化 Leads 与合成路径（树状）

- 输入：`leadgflownet_infer.py` 生成的 JSON，路径 `runs/lead_routes.json`（包含 `leads_set` 与 `routes`）
- 目标：逐个展示 `leads_set` 分子，并树状显示其合成路径（必要时展示多个可能路径）



In [ ]:
import os
import json
from typing import Dict, Any, List

import ipywidgets as widgets
from IPython.display import display, HTML

from rdkit import Chem
from rdkit.Chem import Draw


def _first_existing(paths: List[str]) -> str | None:
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None


def load_json(path: str) -> Dict[str, Any]:
    candidate = _first_existing([
        path,
        'runs/lead_routes.json',
        '../runs/lead_routes.json',
        'lead_routes.json',
        '../lead_routes.json',
    ])
    if candidate is None:
        raise FileNotFoundError(f"JSON not found. Tried: {path}")
    with open(candidate, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"Loaded file: {candidate}")
    return data


def _find_leads_list(obj: Any) -> List[str]:
    # Depth-first search for a key named 'leads_set' (list)
    stack = [obj]
    while stack:
        cur = stack.pop()
        if isinstance(cur, dict):
            if 'leads_set' in cur and isinstance(cur['leads_set'], list):
                return [str(x).strip() for x in cur['leads_set'] if str(x).strip()]
            # Explore children
            for v in cur.values():
                stack.append(v)
        elif isinstance(cur, list):
            for v in cur:
                stack.append(v)
    return []


def mol_svg(smiles: str, size: int = 250) -> str:
    if not smiles:
        return "<div style='color:#888'>—</div>"
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return f"<div style='color:#c00'>Invalid SMILES: {smiles}</div>"
    d2d = Draw.MolDraw2DSVG(size, size)
    d2d.DrawMolecule(mol)
    d2d.FinishDrawing()
    svg = d2d.GetDrawingText()
    return svg


def render_tree(edge_tree: Dict[str, Any], depth: int = 0) -> str:
    # edge_tree is {state, children: [ {block_smiles, rxn_template, result_smiles, subtree}, ... ]}
    state = edge_tree.get('state', '')
    html = []
    html.append(f"<div style='margin-left:{depth*16}px;'>")
    # state row
    html.append("<div style='display:flex; align-items:flex-start; gap:12px; margin:6px 0;'>")
    html.append("<div>")
    html.append("<div style='font-size:12px; color:#666'>当前状态</div>")
    html.append(mol_svg(state, size=160))
    html.append("</div>")
    html.append("</div>")

    # children edges
    children = edge_tree.get('children', [])
    for child in children:
        block = child.get('block_smiles', '')
        rxn = child.get('rxn_template', '')
        result = child.get('result_smiles', '')
        subtree = child.get('subtree', {})

        html.append(f"<div style='margin-left:{(depth)*16}px; padding:8px 0 0 0;'>")
        html.append("<div style='display:flex; align-items:center; gap:12px;'>")
        html.append("<div style='text-align:center;'>")
        html.append("<div style='font-size:12px; color:#666'>Building block</div>")
        html.append(mol_svg(block, size=140))
        html.append("</div>")
        html.append("<div style='font-size: 24px; padding: 0 4px;'>+")
        html.append("</div>")
        html.append("<div style='text-align:center;'>")
        html.append("<div style='font-size:12px; color:#666'>反应模板</div>")
        html.append(f"<div style='max-width:500px; font-family:monospace; font-size:12px; color:#444'>{rxn}</div>")
        html.append("</div>")
        html.append("<div style='font-size: 24px; padding: 0 4px;'>&rarr;")
        html.append("</div>")
        html.append("<div style='text-align:center;'>")
        html.append("<div style='font-size:12px; color:#666'>产物</div>")
        html.append(mol_svg(result, size=160))
        html.append("</div>")
        html.append("</div>")
        html.append("</div>")

        if subtree:
            html.append(render_tree(subtree, depth=depth + 1))

    html.append("</div>")
    return "\n".join(html)


def find_path_to_leaf(node: Dict[str, Any], target: str) -> List[Dict[str, str]] | None:
    children = node.get('children', [])
    for child in children:
        edge = {
            'source_state': node.get('state', ''),
            'block_smiles': child.get('block_smiles', ''),
            'rxn_template': child.get('rxn_template', ''),
            'result_smiles': child.get('result_smiles', ''),
        }
        if edge['result_smiles'] == target:
            return [edge]
        sub = child.get('subtree', {}) or {}
        if sub:
            sub_path = find_path_to_leaf(sub, target)
            if sub_path is not None:
                return [edge] + sub_path
    return None


def path_to_pseudotree(path_edges: List[Dict[str, str]]) -> Dict[str, Any]:
    if not path_edges:
        return {}
    root = {'state': path_edges[0]['source_state'], 'children': []}
    cursor = root
    for edge in path_edges:
        child = {
            'block_smiles': edge['block_smiles'],
            'rxn_template': edge['rxn_template'],
            'result_smiles': edge['result_smiles'],
            'subtree': {'state': edge['result_smiles'], 'children': []}
        }
        cursor['children'].append(child)
        cursor = child['subtree']
    return root


def render_lead_with_routes(data: Dict[str, Any], lead_smi: str) -> str:
    html = []
    html.append("<div style='padding:10px; border:1px solid #ddd; border-radius:8px; margin:12px 0;'>")
    html.append("<div style='margin-bottom:8px; font-weight:bold'>Lead</div>")
    html.append(mol_svg(lead_smi, size=240))

    routes = data.get('routes', [])
    cnt = 0
    for r in routes:
        tree = r.get('tree', {}) or {}
        if not isinstance(tree, dict):
            continue
        path = find_path_to_leaf(tree, lead_smi)
        if path:
            html.append("<div style='margin-top:12px; color:#555'>包含该 lead 的完整路径：</div>")
            pseudo = path_to_pseudotree(path)
            html.append(render_tree(pseudo, depth=0))
            cnt += 1
    if cnt == 0:
        html.append("<div style='color:#888; margin-top:8px;'>未在当前采样的树中发现完整路径（可能在更深层或未分支到该产物）。</div>")
    html.append("</div>")
    return "\n".join(html)


# UI
json_path = widgets.Text(value='../runs/lead_routes.json', description='JSON路径:', layout=widgets.Layout(width='600px'))
load_btn = widgets.Button(description='加载', button_style='primary')
leads_dd = widgets.Dropdown(options=[], description='选择Lead:')
output = widgets.Output()

state = {'data': None}


def on_load_clicked(_):
    with output:
        output.clear_output()
        try:
            data = load_json(json_path.value)
        except Exception as e:
            print('加载失败:', e)
            return
        state['data'] = data

        # Debug summary
        print('Top-level keys:', list(data.keys()))
        routes = data.get('routes', [])
        print('routes count =', len(routes))

        # Try to read leads_set; if missing, search recursively
        leads = data.get('leads_set', [])
        found_direct = isinstance(leads, list) and len(leads) > 0
        if not isinstance(leads, list) or not leads:
            leads = _find_leads_list(data)
        found_recursive = len(leads) > 0
        # Normalize
        leads = [str(x).strip() for x in leads if str(x).strip()]

        # Fallback: derive from routes' leaves
        used_fallback = False
        if not leads:
            try:
                leaf_set = set()
                for r in routes:
                    tree = r.get('tree', {})
                    def dfs(node):
                        children = node.get('children', [])
                        if not children:
                            return
                        for c in children:
                            subtree = c.get('subtree', {})
                            # Treat edges whose subtree has no children as leaves
                            if not subtree or not subtree.get('children'):
                                smi = str(c.get('result_smiles', '')).strip()
                                if smi:
                                    leaf_set.add(smi)
                            else:
                                dfs(subtree)
                    dfs(tree)
                leads = sorted(leaf_set)
                used_fallback = True
            except Exception as e:
                print('derive-from-routes failed:', e)
                leads = []

        # Print diagnostics
        print('leads_set direct?', found_direct, '| recursive?', found_recursive, '| fallback_used?', used_fallback)
        print('leads count =', len(leads))
        print('first 10 leads:', leads[:10])
        # Route children stats (first level)
        empty_children = 0
        sample_report = []
        for i, r in enumerate(routes[:10]):
            tree = r.get('tree', {})
            nchild = len(tree.get('children', [])) if isinstance(tree, dict) else 0
            if nchild == 0:
                empty_children += 1
            sample_report.append((i, nchild))
        print('first 10 routes first-level children counts:', sample_report)

        leads_dd.options = leads
        if leads:
            leads_dd.value = leads[0]
            print(f"Loaded. leads={len(leads)} routes={len(routes)}")
        else:
            print(f"Loaded. No leads found. routes={len(routes)}")


def on_lead_change(change):
    if change['name'] != 'value':
        return
    lead = change['new']
    data = state.get('data')
    if not data or not lead:
        return
    with output:
        output.clear_output()
        html = render_lead_with_routes(data, lead)
        display(HTML(html))


load_btn.on_click(on_load_clicked)
leads_dd.observe(on_lead_change)

widgets.VBox([
    widgets.HBox([json_path, load_btn]),
    leads_dd,
    output,
])
